# 06 · Score the answers honestly

**Expected JSON + actual JSON → field checks → score**

Use three held-out practice examples. Report malformed answers and wrong labels as failures. This is a development check, not a final benchmark.

**Who does what:** Nano runs the evaluation tools. The CPU candidate and both Lightning aliases produce answers. Python scores them against known labels.

###  evaluation contracts and reproducibility

NeMo Evaluator's [quickstart](https://docs.nvidia.com/nemo/evaluator/get-started/quickstart) and [custom benchmark tutorial](https://docs.nvidia.com/nemo/evaluator/tutorials/byob) show how to separate datasets, model execution, scoring, and reports. They are extension references: this notebook currently uses the Python scoring cells below, not NeMo Evaluator.

load the CPU candidate from 04 and the validation data from 02; inspect its generated answers. Then compare Lightning base/rules, base/full-taxonomy, and specialist/rules on the same three records. Live comparisons and adapter metadata require the [provisioned services/assets](../README.md#provision-services-before-running-the-course).

**What counts as success:** parseable JSON, correct fields, and a wholly correct answer are different measurements. A completed run can have poor accuracy. Three development cases teach scoring; they do not establish generalization. Preserve prompts, generation settings, dataset hash, and the actual model identity.

**Next step:** [Evaluator integration notes](README.md#evaluation-references) explain the separate environment, custom scorer, and distinction between agent instruction skills and the NeMo Skills benchmark library. No Evaluator package or skill is installed by this notebook.

**NVIDIA evaluation boundary:** the linked NeMo Evaluator documentation describes a possible extension. Our measured outputs come from the local Python scorer, not an Evaluator service or NVIDIA benchmark. The three cases remain development checks.


**Manual experiment versus agent reference:** the executable cells below are yours to edit. The OpenClaw section at the end runs the fixed workshop recipe in `agent/lessons/`; it does not ingest your edited cells or reuse manual artifacts. Setup, displays, artifact bookkeeping, and terminal/process transport remain helpers.

### Start here
Run the executable cells in order. Edit the visible requests, rules, and experiment parameters to change your manual run. The artifact helper only copies prerequisites and records output paths. Starting again creates a fresh output folder.

In [ ]:
from pathlib import Path
import sys, json
from IPython.display import JSON, display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course/common.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.agent.lesson_walkthrough import table, bars, loss_chart, show_skill_library
from course.agent.notebook_support import NotebookRun
from course.common import chat, request, rows, BASE_URL
import os, time

lesson = NotebookRun('06-evaluate')
OUTPUT = lesson.work
LIVE = True


### 1 · Test the CPU candidate

The tiny model may score **0/3**. That is a valid evaluation result, not a reason to hide the output.

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel
BUILD = OUTPUT/'cpu'
student = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(BUILD/'base',local_files_only=True),
    BUILD/'student-adapter',is_trainable=False)
student.eval(); torch.set_num_threads(2)
fresh_validation = [json.loads(line) for line in (OUTPUT/'student-data/validation.jsonl').read_text().splitlines()]
student_results = []
for index,row in enumerate(fresh_validation):
    prompt = 'Observation: '+row['messages'][1]['content']+'\nAnswer: '
    ids = torch.tensor([[b+2 for b in prompt.encode()]])
    with torch.no_grad():
        output = student.generate(ids,attention_mask=torch.ones_like(ids),max_new_tokens=160,
                                  do_sample=False,pad_token_id=0,eos_token_id=1)
    text = bytes([x-2 for x in output[0,len(ids[0]):].tolist() if 2<=x<258]).decode(errors='replace')
    try:
        prediction = json.loads(text)
        parsed = isinstance(prediction,dict)
    except ValueError:
        prediction,parsed = None,False
    gold = json.loads(row['messages'][-1]['content'])
    student_results.append({'example':index,'generated':text,'parsed_json':parsed,'exact_match':prediction==gold})
print('Your candidate valid JSON:',sum(r['parsed_json'] for r in student_results),'/',len(student_results))
print('Your candidate exact match:',sum(r['exact_match'] for r in student_results),'/',len(student_results))
(OUTPUT/'student-candidate-evaluation.json').write_text(json.dumps(student_results,indent=2))

In [ ]:
candidate = student_results
table(["Example", "Valid JSON", "Exact match", "Generated answer"], [(r["example"], r["parsed_json"], r["exact_match"], r["generated"]) for r in candidate])

### 2 · Compare model and prompt choices

| Arm | Model | Prompt |
|---|---|---|
| base_rules | Lightning base | Rules only |
| base_full | Lightning base | Rules + full category guidance |
| specialist_rules | Lightning + adapter | Rules only |

Run nine fresh requests: three examples for each arm. This distinguishes a prompt change from an adapter change.

In [ ]:
from train.scripts.jisr_rules import RULES_PROMPT, SYSTEM_PROMPT
FIELDS = ('activity_type','sensor_quality','confidence','routing')
examples = [json.loads(line) for line in (OUTPUT/'student-data/validation.jsonl').read_text().splitlines()]
arms = [('base','base_rules',RULES_PROMPT),('base','base_full',SYSTEM_PROMPT),
        ('specialist','specialist_rules',RULES_PROMPT)]
results = []
if LIVE:
    for model, arm, prompt in arms:
        for index, row in enumerate(examples):
            gold = json.loads(row['messages'][-1]['content'])
            try:
                raw = chat(model,[{'role':'system','content':prompt},row['messages'][1]],json_output=True)
                prediction = json.loads(raw)
                if not isinstance(prediction,dict): raise ValueError('Not an object')
                correct = {f:prediction.get(f)==gold[f] for f in FIELDS}
                result = {'arm':arm,'example':index,'gold':gold,'prediction':prediction,'correct':correct}
            except (ValueError, OSError) as exc:
                result = {'arm':arm,'example':index,'gold':gold,'error':str(exc),
                          'correct':{f:False for f in FIELDS}}
            results.append(result)
    (OUTPUT/'fresh-evaluation.json').write_text(json.dumps(results,indent=2))
else:
    print('SKIPPED: fresh evaluation. No recorded scores substituted.')

In [ ]:
for _, arm, _ in arms:
    subset = [r for r in results if r['arm']==arm]
    if subset:
        print(arm, 'fully correct:', sum(all(r['correct'].values()) for r in subset), '/',len(subset))
        print('Fields:', {f:sum(r['correct'][f] for r in subset)/len(subset) for f in FIELDS})
for row in results:
    if not all(row['correct'].values()): print(json.dumps(row,indent=2))

In [ ]:
results = results
scores = {arm: sum(all(r["correct"].values()) for r in results if r["arm"] == arm) for _,arm,_ in arms}
bars(scores, maximum=3)
print("Each bar is fully correct answers out of 3.")

### 3 · Inspect an answer

A score tells you where to look. The actual prediction and expected label show what went wrong. Change the index to inspect a different result.

In [ ]:
example_index = 0
display(JSON(results[example_index], expanded=True))

### 4 · Save the evidence

Save the dataset hash, adapter configuration, and inference settings. **PASS means the evaluation completed; it does not mean the model passed an accuracy threshold.**

In [ ]:
import hashlib
manifest = {'dataset':'course/outputs/student-data/validation.jsonl','selection':'first 3, development only',
 'dataset_sha256':hashlib.sha256((OUTPUT/'student-data/validation.jsonl').read_bytes()).hexdigest(),
 'adapter_config':json.loads((ROOT/'train/adapters/jisr-lightning/adapter_config.json').read_text()),
 'temperature':0,'max_tokens':512,'structured_json':True,'live_executed':LIVE}
(OUTPUT/'evaluation-manifest.json').write_text(json.dumps(manifest,indent=2))
print(json.dumps(manifest,indent=2))

In [ ]:
manifest = lesson.read("evaluation-manifest.json")
manifest["dataset"] = str(lesson.work / "student-data/validation.jsonl")
(lesson.work / "evaluation-manifest.json").write_text(json.dumps(manifest, indent=2))
display(JSON(manifest, expanded=False))

### Check and save this manual run
Validate the artifacts and record this run for the next manual walkthrough. A successful check describes execution and evidence; read the actual quality scores separately.

In [ ]:
assert len(student_results) == len(fresh_validation)
assert len(results) == len(arms) * len(examples)
manual_result = lesson.finish({'candidate_exact_match': sum(r['exact_match'] for r in student_results),
    'candidate_examples': len(student_results), 'endpoint_results': results,
    'meaning': 'Execution passed; inspect accuracy separately'})
display(JSON(manual_result, expanded=False))

### Grow the skill library
The library now includes this lesson’s skill, **workshop-evaluate**. Open the actual instructions below.

**`inspect-cases` → `evaluate` → `check-results`**

In [ ]:
show_skill_library('06-evaluate')

### Prepare OpenClaw
Run once for each new demo. This cell installs the growing skill library in a fresh sandbox workspace and starts a background host worker for this lesson’s three actions. **It does not ask the agent to do anything yet.**

Here, **installing a skill means placing its `SKILL.md` instruction file in OpenClaw’s workspace**. The current lesson’s source is `course/agent/skills/workshop-evaluate/SKILL.md` in this repository.

Each run gets this structure inside the OpenShell sandbox (launcher files omitted):

```text
/home/sandbox/course-runs/openclaw-<run-id>/
├── openclaw.json                 # points OpenClaw at this run’s workspace
├── chapter_tool.py              # helper invoked by the agent
├── chapter.json                 # this lesson’s task and allowed actions
└── workspace/
    ├── AGENTS.md                 # workspace instructions
    └── skills/
        ├── workshop-evaluate/
        │   └── SKILL.md
        └── ...                  # skills from earlier notebooks
```

OpenClaw discovers skills under `workspace/skills/`. The task you paste in the next step tells the agent which skill to read and supplies the full path to `chapter_tool.py` (the skill’s `<tool_path>` placeholder).

The agent invokes the helper through its execution tool. The helper forwards the selected action to the host worker, which runs the fixed lesson code and returns the result. Nano follows the skill and coordinates the calls.

Complete these earlier **OpenClaw demos** first: `02-training-data`, `03-build-lora`, `04-train-adapter`. This run copies their latest verified agent artifacts; its inspect action reports the inputs.

Start the TUI task within **15 minutes** and keep this run’s `terminal_run` value for collecting results. If the worker expires, prepare a new run. The local run directory stores host results; the installed skills live at the sandbox paths printed below.


In [ ]:
from course.agent.chapter_terminal import prepare_chapter_terminal, collect_chapter_terminal
from course.agent.notebook_terminal import open_terminal
terminal_run = prepare_chapter_terminal('06-evaluate')
print("Local run directory:", terminal_run["directory"])
print("OpenClaw sandbox workspace:", terminal_run["remote"] + "/workspace")
print("Installed skill:", terminal_run["remote"] + "/workspace/skills/workshop-evaluate/SKILL.md")
print("Python helper:", terminal_run["remote"] + "/chapter_tool.py")
print("\nThe agent will run these commands individually:")
for action in ['inspect-cases', 'evaluate', 'check-results']:
    print("python3 " + terminal_run["remote"] + "/chapter_tool.py " + action)

### Open the TUI and paste the task
Exit the previous TUI with `/exit` first. This cell opens a terminal on the workshop computer’s desktop. Copy only the generated task into OpenClaw.

The terminal label is **Nano**, its lightweight orchestration model. Nano runs the evaluation tools. The CPU candidate and both Lightning aliases produce answers. Python scores them against known labels.

In [ ]:
open_terminal(terminal_run)
print("\nPASTE THIS INTO OPENCLAW:\n")
print(terminal_run["prompt"])

### Bring the evidence back
After the TUI finishes, run this cell. It retrieves that session’s trace and checks the artifacts and skill reads. If Run All reaches this cell before you finish the TUI task, run it again afterward.

**Look for:** Candidate score: 0/3 can be a valid result; it means the model needs work.

In [ ]:
agent_result = collect_chapter_terminal(terminal_run)
display(JSON(agent_result, expanded=True))

**What you demonstrated:** Choose the next improvement from actual errors, then retest on new examples.

Exit the TUI with `/exit` when done.